# Лабораторная: наивный Байес на «Квартирах»

Капстоун модуля III «Вероятность». Уроки III.2, III.3, III.12 и III.13. Около 60 минут.

[Открыть в Colab](https://colab.research.google.com/github/perewodchik/Machine-course/blob/main/notebooks/prob/naive_bayes_apartments.ipynb)

**После лабораторной вы умеете:**
- оценивать параметры наивного Байеса максимумом правдоподобия — долями;
- предсказывать класс через сумму логарифмов, а не произведение вероятностей;
- объяснять и исправлять нулевые оценки сглаживанием Лапласа;
- сравнивать модели по кросс-энтропии на отложенной выборке;
- находить информативные признаки через KL-дивергенцию.

In [ ]:
# Настройка: пакет курса с проверками. Локально и в CI он уже установлен, в Colab ставится из GitHub.
try:
    from mlf import check
except ImportError:
    %pip install -q "mlf @ git+https://github.com/perewodchik/Machine-course#subdirectory=code"
    from mlf import check
import numpy as np

## Данные

$600$ объявлений о продаже квартир. Признаки — есть ли в тексте каждое из семи слов: $x_j^{(i)} \in \{0, 1\}$. Метка $y^{(i)} = 1$, если квартира продалась за неделю. Первые $400$ объявлений — обучающая выборка, остальные $200$ — отложенная.

Модель (CS229, «Generative learning algorithms», «Naive Bayes»): $y \sim \operatorname{Bernoulli}(\phi_y)$, и при известном $y$ слова **условно независимы** (урок III.2): $p(x \mid y) = \prod_j p(x_j \mid y)$, $x_j \mid y \sim \operatorname{Bernoulli}(\phi_{j \mid y})$.

In [ ]:
# из code/snippets/prob/naive_bayes.py, регион data
WORDS = ("метро", "ремонт", "срочно", "торг", "новостройка", "без посредников", "аукцион")


def apartment_ads(n: int = 600, seed: int = 2027):
    """Объявления: X — n × 7 из 0 и 1 (есть ли слово), y — продалась ли квартира за неделю."""
    rng = np.random.default_rng(seed)
    y = (rng.random(n) < 0.35).astype(int)
    phi1 = np.array([0.70, 0.60, 0.50, 0.40, 0.30, 0.50, 0.01])
    phi0 = np.array([0.40, 0.30, 0.10, 0.30, 0.30, 0.45, 0.04])
    phi = np.where(y[:, None] == 1, phi1, phi0)
    X = (rng.random((n, len(WORDS))) < phi).astype(int)
    return X, y


X, y = apartment_ads()
X_train, y_train, X_test, y_test = X[:400], y[:400], X[400:], y[400:]

print("слова:", WORDS)
print("X_train:", X_train.shape, " доля проданных:", y_train.mean())

## Задание 1. Оценки максимального правдоподобия

Напишите `fit(X, y, alpha=0)`: возвращает `phi_y` — долю $y = 1$, и два вектора длины $7$: `phi1[j]` — долю объявлений со словом $j$ среди $y = 1$, `phi0[j]` — среди $y = 0$. Параметр `alpha` пока равен нулю (задание 4). Это максимум правдоподобия: для Бернулли оценка — доля (урок II.8).

In [ ]:
def fit(X, y, alpha=0.0):
    # Ваш код здесь
    raise NotImplementedError


phi_y, phi1, phi0 = fit(X_train, y_train)
check.shape(phi1, (7,), name="phi1")
check.close(phi_y, 0.3375, name="phi_y")
check.close(phi1[0], np.mean(X_train[y_train == 1, 0]), name="phi1 для «метро»")
for w, a, b in zip(WORDS, phi1, phi0):
    print(f"{w:>15}: P(слово | продалась) = {a:.3f}, P(слово | нет) = {b:.3f}")
check.passed()

## Задание 2. Предсказание через логарифмы

Апостериорная вероятность по формуле Байеса (урок III.3):
$$P(y = 1 \mid x) = \frac{\phi_y \prod_j p(x_j \mid 1)}{\phi_y \prod_j p(x_j \mid 1) + (1 - \phi_y) \prod_j p(x_j \mid 0)} .$$
Произведение многих вероятностей уходит в машинный ноль, поэтому считают логарифмы: $a_c = \log P(y = c) + \sum_j \log p(x_j \mid c)$, и $P(y = 1 \mid x) = 1 / (1 + e^{a_0 - a_1})$ — та же сигмоида, что в логистической регрессии.

Напишите `log_joint(X, phi_y, phi1, phi0)` — массив `n × 2` со столбцами $a_0$ и $a_1$, и `predict_proba` — вектор $P(y = 1 \mid x^{(i)})$. Используйте `np.errstate(divide="ignore")`: логарифм нуля пусть будет $-\infty$. Осторожно: `X @ np.log(phi)` при нулевой оценке даст $0 \cdot (-\infty) = $ `nan` — берите `np.where`.

In [ ]:
def log_joint(X, phi_y, phi1, phi0):
    # Ваш код здесь
    raise NotImplementedError


def predict_proba(X, phi_y, phi1, phi0):
    a = log_joint(X, phi_y, phi1, phi0)
    # Ваш код здесь
    raise NotImplementedError


x = X_test[:1]
direct = phi_y * np.prod(np.where(x == 1, phi1, 1 - phi1)) / (phi_y * np.prod(np.where(x == 1, phi1, 1 - phi1)) + (1 - phi_y) * np.prod(np.where(x == 1, phi0, 1 - phi0)))
check.shape(log_joint(X_test, phi_y, phi1, phi0), (200, 2), name="log_joint")
check.close(predict_proba(x, phi_y, phi1, phi0)[0], direct, name="P(y = 1 | x) против прямой формулы")
check.passed()

## Задание 3. Точность и нулевая оценка

Посчитайте `accuracy` — долю верных ответов на отложенной выборке при пороге $0.5$, и `baseline` — точность константы «не продастся». Посмотрите на объявления со словом «аукцион».

In [ ]:
proba = predict_proba(X_test, phi_y, phi1, phi0)
# Ваш код здесь
raise NotImplementedError
print(f"точность {accuracy:.3f}, константа {baseline:.3f}")
auction = WORDS.index("аукцион")
print("P(y = 1 | x) у объявлений с «аукционом»:", proba[X_test[:, auction] == 1].round(3))
assert accuracy > baseline, "модель должна быть лучше константы"
check.passed()

**Вопрос для самопроверки.** Почему у всех объявлений со словом «аукцион» вероятность продажи ровно $0$, какие бы другие слова в них ни были? Что в обучающей выборке привело к `phi1[6] = 0`, и разумно ли из одной ненаблюдавшейся комбинации делать вывод «никогда»?

## Задание 4. Сглаживание Лапласа

Добавьте к каждому счётчику $\alpha$: $\hat\phi_{j \mid y} = \dfrac{\#\{x_j = 1, y\} + \alpha}{\#\{y\} + 2\alpha}$. При $\alpha = 1$ это **сглаживание Лапласа** — максимум апостериорной плотности с априорным $\operatorname{Beta}(2, 2)$ (уроки III.7 и IV.6). Допишите `fit` выше так, чтобы он учитывал `alpha`, и проверьте: ни одна оценка не равна $0$ или $1$, а вероятности у объявлений с «аукционом» стали осмысленными.

In [ ]:
s_phi_y, s_phi1, s_phi0 = fit(X_train, y_train, alpha=1.0)
assert np.all((s_phi1 > 0) & (s_phi1 < 1)) and np.all((s_phi0 > 0) & (s_phi0 < 1)), "после сглаживания оценки строго между 0 и 1"
check.close(s_phi1[auction], 1 / (np.sum(y_train == 1) + 2), name="phi1 для «аукциона» после сглаживания")
s_proba = predict_proba(X_test, s_phi_y, s_phi1, s_phi0)
print("с «аукционом» после сглаживания:", s_proba[X_test[:, auction] == 1].round(3))
check.passed()

## Задание 5. Кросс-энтропия на отложенной выборке

Точность не видит уверенности. Напишите `log_loss(y, p)` — среднюю кросс-энтропию $-\frac1n \sum_i \bigl(y^{(i)} \ln p^{(i)} + (1 - y^{(i)}) \ln (1 - p^{(i)})\bigr)$ (урок III.13). Сравните три модели: без сглаживания, со сглаживанием и константу $p^{(i)} = \hat\phi_y$ — у константы кросс-энтропия равна кросс-энтропии меток и доли $\hat\phi_y$.

In [ ]:
def log_loss(y, p):
    # Ваш код здесь
    raise NotImplementedError


raw, smooth, const = log_loss(y_test, proba), log_loss(y_test, s_proba), log_loss(y_test, np.full(200, s_phi_y))
print(f"log loss: без сглаживания {raw}, со сглаживанием {smooth:.4f}, константа {const:.4f}")
assert np.isinf(raw) or raw > smooth, "нулевая вероятность у реального события даёт бесконечную потерю"
assert smooth < const, "модель должна быть лучше константы и по кросс-энтропии"
check.passed()

## Задание 6. Какие слова информативны

Слово полезно, если его распределение у проданных и непроданных разное. Меру даёт KL-дивергенция двух Бернулли: $D_{KL}\bigl(\operatorname{Bern}(\phi_{j \mid 1}) \,\Vert\, \operatorname{Bern}(\phi_{j \mid 0})\bigr)$. Посчитайте её для каждого слова по сглаженным оценкам и найдите самое и наименее информативное.

In [ ]:
# Ваш код здесь
raise NotImplementedError
order = np.argsort(kl)[::-1]
for j in order:
    print(f"{WORDS[j]:>15}: KL = {kl[j]:.4f}")
assert WORDS[order[0]] == "срочно", f"самое информативное слово — не {WORDS[order[0]]!r}"
assert WORDS[order[-1]] == "новостройка", f"наименее информативное — не {WORDS[order[-1]]!r}"
check.passed()

**Вопрос для самопроверки.** «Новостройка» встречается в $30\,\%$ объявлений обоих классов. Почему её KL почти ноль и почему её можно выбросить из модели без потерь? И почему наивный Байес, вопреки названию, неплохо работает, даже когда слова на самом деле зависимы?